## 1. NDVI Analysis and Visualization

This notebook analyzes the AOI-cropped Sentinel-2 NDVI GeoTIFFs produced in the previous processing stage. It combines raster observations with the soil–vegetation analysis layer to examine spatial and temporal NDVI patterns across the GeoSoil-Cuba Jaruco/Mayabeque area of interest (AOI).

The workflow validates raster metadata and acquisition dates, summarizes NDVI values, visualizes spatial and temporal patterns, screens observations using Sentinel-2 Scene Classification Layer (SCL) data, and evaluates a combined agricultural analysis region. Outputs are saved under the project `outputs/tables` and `outputs/figures` directories for later use in reports, PDFs, and presentations.

The analysis reuses existing NDVI products and avoids additional processing unless required for validation or quality control.

In [ ]:
# ---------------------------------------------------------
# 1. Paths and NDVI raster inventory
# ---------------------------------------------------------

from pathlib import Path
import re
import pandas as pd
import rasterio
import numpy as np
import geopandas as gpd


# Project and output paths
PROJECT_DIR = Path("..").resolve()

soil_vegetation_path = (
    PROJECT_DIR / "data" / "processed" / "vector" / "soil_vegetation_analysis.gpkg"
)

ndvi_aoi_dir = (
    PROJECT_DIR / "data" / "processed" / "satellite" / "ndvi_aoi"
)

OUTPUTS_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# Find all NDVI GeoTIFF files
ndvi_files = sorted(ndvi_aoi_dir.glob("*.tif"))


print(f"Soil–vegetation layer: {soil_vegetation_path}")
print(f"NDVI directory:        {ndvi_aoi_dir}")
print(f"NDVI rasters found:    {len(ndvi_files)}")


# Display the raster filenames
for path in ndvi_files:
    print(path.name)


layers = gpd.list_layers(soil_vegetation_path)

for layer in layers["name"]:
    print(layer)

## 2. Inspect the NDVI raster metadata

Before calculating statistics or plotting the rasters, we should verify that all GeoTIFFs are spatially consistent. In particular, we want to confirm that they share the same CRS, resolution, dimensions, transform, NoData value, and band structure.

In [ ]:
# ---------------------------------------------------------
# 2b. Display raster metadata with explicit spatial names
# ---------------------------------------------------------

raster_records = []

for path in ndvi_files:

    with rasterio.open(path) as src:

        raster_records.append({
            "file": path.name,
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "crs": src.crs.to_string(),
            "resolution_x": src.res[0],
            "resolution_y": src.res[1],
            "nodata": src.nodata,
            "dtype": src.dtypes[0],
            "left": src.bounds.left,
            "bottom": src.bounds.bottom,
            "right": src.bounds.right,
            "top": src.bounds.top
        })


ndvi_metadata = pd.DataFrame(raster_records)

ndvi_metadata

## 3. Check whether all rasters have identical spatial grids

In [ ]:
# ---------------------------------------------------------
# 3. Verify that all NDVI rasters share the same grid
# ---------------------------------------------------------

reference = ndvi_files[0]

with rasterio.open(reference) as src:
    reference_crs = src.crs
    reference_transform = src.transform
    reference_width = src.width
    reference_height = src.height
    reference_res = src.res
    reference_nodata = src.nodata


grid_checks = []

for path in ndvi_files:

    with rasterio.open(path) as src:

        grid_checks.append({
            "file": path.name,
            "same_crs": src.crs == reference_crs,
            "same_transform": src.transform == reference_transform,
            "same_dimensions": (
                src.width == reference_width
                and src.height == reference_height
            ),
            "same_resolution": src.res == reference_res,
            "same_nodata": src.nodata == reference_nodata
        })


grid_check = pd.DataFrame(grid_checks)

grid_check

## 4. Extract acquisition dates and build the NDVI time series inventory

The next step is to associate each raster with its acquisition date and create a clean inventory. We should derive the date from the filename rather than manually entering dates.

In [ ]:
# ---------------------------------------------------------
# 4. Extract acquisition dates and build NDVI inventory
# ---------------------------------------------------------

ndvi_records = []

for path in ndvi_files:

    # Search for an ISO date in the filename
    match = re.search(r"\d{4}-\d{2}-\d{2}", path.name)

    if match is None:
        raise ValueError(f"No acquisition date found in: {path.name}")

    acquisition_date = pd.to_datetime(match.group())

    ndvi_records.append({
        "acquisition_date": acquisition_date,
        "file": path.name,
        "path": path
    })


ndvi_inventory = (
    pd.DataFrame(ndvi_records)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)


ndvi_inventory
ndvi_inventory.to_csv(TABLES_DIR / "stage5_ndvi_inventory.csv", index=False)

## 5. Check the temporal coverage

In [ ]:
# ---------------------------------------------------------
# 5. Summarize the temporal coverage of the NDVI series
# ---------------------------------------------------------

print(
    f"First acquisition: "
    f"{ndvi_inventory['acquisition_date'].min():%Y-%m-%d}"
)

print(
    f"Last acquisition:  "
    f"{ndvi_inventory['acquisition_date'].max():%Y-%m-%d}"
)

print(
    f"Number of observations: "
    f"{len(ndvi_inventory)}"
)


# Calculate the interval between consecutive observations
ndvi_inventory["days_since_previous"] = (
    ndvi_inventory["acquisition_date"]
    .diff()
    .dt.days
)


ndvi_inventory

## 6. Calculate basic NDVI statistics for every acquisition

Now we can inspect the actual NDVI values. This is an important validation step before visualization because it tells us whether the rasters contain plausible NDVI ranges and whether any acquisition has an unusual amount of valid or invalid data.

In [ ]:
# ---------------------------------------------------------
# 6. Calculate basic statistics for each NDVI raster
# ---------------------------------------------------------

stats = []

for _, row in ndvi_inventory.iterrows():

    with rasterio.open(row["path"]) as src:

        data = src.read(1, masked=True)

        stats.append({
            "acquisition_date": row["acquisition_date"],
            "file": row["file"],
            "valid_pixels": int(data.count()),
            "mean_ndvi": float(data.mean()),
            "median_ndvi": float(np.ma.median(data)),
            "min_ndvi": float(data.min()),
            "max_ndvi": float(data.max()),
            "std_ndvi": float(data.std())
        })


ndvi_stats = (
    pd.DataFrame(stats)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)


display(ndvi_stats)
ndvi_stats.to_csv(TABLES_DIR / "stage5_ndvi_basic_statistics.csv", index=False)

## 7. Inspect the extreme NDVI values

Several dates contain exact -1.0 and/or 1.0 NDVI values. Those are mathematically possible NDVI limits, but the fact that some dates have them while others do not makes it worth determining whether they represent real pixels or residual edge/cloud/water effects.

In [ ]:
# ---------------------------------------------------------
# 7. Inspect pixels at the theoretical NDVI limits
# ---------------------------------------------------------

extreme_stats = []

for _, row in ndvi_inventory.iterrows():

    with rasterio.open(row["path"]) as src:

        data = src.read(1, masked=True)

        extreme_stats.append({
            "acquisition_date": row["acquisition_date"],
            "ndvi_minus_1_pixels": int(np.sum(data == -1.0)),
            "ndvi_plus_1_pixels": int(np.sum(data == 1.0)),
            "pixels_below_0": int(np.sum(data < 0)),
            "pixels_above_0_7": int(np.sum(data > 0.7))
        })


ndvi_extremes = (
    pd.DataFrame(extreme_stats)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)


display(ndvi_extremes)
ndvi_extremes.to_csv(TABLES_DIR / "stage5_ndvi_extreme_value_check.csv", index=False)

## 8. Visualize the temporal mean NDVI

Assuming the extreme-value check does not reveal a major problem, the first actual visualization should be the mean NDVI through time. This gives us the broad seasonal signal before we examine spatial patterns.

In [ ]:
# ---------------------------------------------------------
# 8. Plot mean NDVI through time
# ---------------------------------------------------------

import matplotlib.pyplot as plt


plt.figure(figsize=(11, 5))

plt.plot(
    ndvi_stats["acquisition_date"],
    ndvi_stats["mean_ndvi"],
    marker="o"
)

plt.xlabel("Acquisition date")
plt.ylabel("Mean NDVI")
plt.title("Mean NDVI Through Time — GeoSoil-Cuba AOI")

plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_15.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Compare mean and median NDVI through time

Because the mean and median can respond differently to a small number of extreme pixels, comparing them gives us a useful robustness check before moving into spatial analysis.

In [ ]:
# ---------------------------------------------------------
# 9. Compare mean and median NDVI through time
# ---------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.plot(
    ndvi_stats["acquisition_date"],
    ndvi_stats["mean_ndvi"],
    marker="o",
    label="Mean NDVI"
)

plt.plot(
    ndvi_stats["acquisition_date"],
    ndvi_stats["median_ndvi"],
    marker="o",
    label="Median NDVI"
)

plt.xlabel("Acquisition date")
plt.ylabel("NDVI")
plt.title("Mean and Median NDVI Through Time — GeoSoil-Cuba AOI")

plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_17.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Calculate the mean–median difference

In [ ]:
# ---------------------------------------------------------
# 10. Calculate the difference between mean and median NDVI
# ---------------------------------------------------------

ndvi_stats["mean_median_difference"] = (
    ndvi_stats["mean_ndvi"] -
    ndvi_stats["median_ndvi"]
)


ndvi_stats[
    [
        "acquisition_date",
        "mean_ndvi",
        "median_ndvi",
        "mean_median_difference"
    ]
]

display(ndvi_stats)

ndvi_stats[["acquisition_date", "mean_ndvi", "median_ndvi", "mean_median_difference"]].to_csv(TABLES_DIR / "stage5_ndvi_mean_median_difference.csv", index=False)

## 11. Visualize the NDVI distribution for each acquisition

Boxplots show the distribution and variability of NDVI for each acquisition much better than mean and median alone.

In [ ]:
# ---------------------------------------------------------
# 11. Create NDVI distribution boxplots for all acquisitions
# ---------------------------------------------------------

ndvi_distributions = []

for _, row in ndvi_inventory.iterrows():

    with rasterio.open(row["path"]) as src:

        data = src.read(1, masked=True)

        ndvi_values = data.compressed()

        ndvi_distributions.append(
            pd.DataFrame({
                "acquisition_date": row["acquisition_date"],
                "ndvi": ndvi_values
            })
        )


ndvi_distribution_df = pd.concat(
    ndvi_distributions,
    ignore_index=True
)


ndvi_distribution_df["date_label"] = (
    ndvi_distribution_df["acquisition_date"]
    .dt.strftime("%Y-%m-%d")
)


plt.figure(figsize=(13, 6))

ndvi_distribution_df.boxplot(
    column="ndvi",
    by="date_label",
    grid=False
)

plt.suptitle("")
plt.title("NDVI Distribution by Acquisition")
plt.xlabel("Acquisition date")
plt.ylabel("NDVI")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_21.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Create a representative NDVI map

Let's now move from temporal statistics to spatial visualization. We'll start with the first acquisition, November 3, 2025, and use the actual NDVI values without rescaling or classification.

In [ ]:
# ---------------------------------------------------------
# 12. Visualize the first NDVI acquisition
# ---------------------------------------------------------

first_ndvi_path = ndvi_inventory.loc[0, "path"]
first_ndvi_date = ndvi_inventory.loc[0, "acquisition_date"]


with rasterio.open(first_ndvi_path) as src:
    ndvi = src.read(1, masked=True)


plt.figure(figsize=(10, 7))

plt.imshow(
    ndvi,
    vmin=-1,
    vmax=1
)

plt.colorbar(
    label="NDVI"
)

plt.title(
    f"NDVI — {first_ndvi_date:%Y-%m-%d}"
)

plt.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_23.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Add the AOI/soil–vegetation layer as spatial context

The next map should connect the satellite result to the GIS data we prepared earlier. First we'll load soil_vegetation_analysis.gpkg and inspect its CRS and geometry count.

In [ ]:
# ---------------------------------------------------------
# 13. Load and inspect the soil–vegetation analysis layer
# ---------------------------------------------------------

import geopandas as gpd


soil_vegetation = gpd.read_file(
    soil_vegetation_path
)


print("Features:", len(soil_vegetation))
print("CRS:", soil_vegetation.crs)
print("Geometry types:")
print(soil_vegetation.geom_type.value_counts())

print("\nColumns:")
print(soil_vegetation.columns.tolist())

## 14. Reproject the soil–vegetation layer to the NDVI CRS

In [ ]:
# ---------------------------------------------------------
# 14. Reproject soil–vegetation layer to NDVI CRS
# ---------------------------------------------------------

with rasterio.open(first_ndvi_path) as src:
    ndvi_crs = src.crs


soil_vegetation_ndvi_crs = soil_vegetation.to_crs(ndvi_crs)


print("NDVI CRS:", ndvi_crs)
print("Vector CRS:", soil_vegetation_ndvi_crs.crs)

## 15. Overlay soil–vegetation polygons on the NDVI map

The NDVI raster is displayed as the continuous background, while the soil–vegetation analysis polygons are overlaid with transparent fill and visible boundaries. The raster and vector data are already aligned in EPSG:32617.

In [ ]:
import matplotlib.pyplot as plt
import rasterio

with rasterio.open(first_ndvi_path) as src:

    ndvi = src.read(1, masked=True)
    bounds = src.bounds

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

fig, ax = plt.subplots(figsize=(10, 7))

# NDVI background
image = ax.imshow(
    ndvi,
    extent=extent,
    origin="upper",
    vmin=-1,
    vmax=1,
    cmap="viridis"
)

# Soil–vegetation polygons
soil_vegetation_ndvi_crs.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.6
)

plt.colorbar(
    image,
    ax=ax,
    label="NDVI"
)

ax.set_title(f"NDVI with Soil–Vegetation Zones — {first_ndvi_date:%Y-%m-%d}")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_29.png", dpi=200, bbox_inches="tight")
plt.show()

## 16. Add polygon IDs to the NDVI map

Each soil–vegetation polygon is labeled with a unique ID. These IDs allow individual regions to be identified visually and then selected for detailed NDVI analysis.

In [ ]:
import matplotlib.pyplot as plt
import rasterio

# Create a sequential polygon ID
soil_vegetation_ndvi_crs = soil_vegetation_ndvi_crs.copy()
soil_vegetation_ndvi_crs["polygon_id"] = range(
    1,
    len(soil_vegetation_ndvi_crs) + 1
)

with rasterio.open(first_ndvi_path) as src:

    ndvi = src.read(1, masked=True)
    bounds = src.bounds

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

fig, ax = plt.subplots(figsize=(12, 8))

# NDVI background
image = ax.imshow(
    ndvi,
    extent=extent,
    origin="upper",
    vmin=-1,
    vmax=1,
    cmap="viridis"
)

# Polygon boundaries
soil_vegetation_ndvi_crs.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.6
)

# Polygon numbers
for _, row in soil_vegetation_ndvi_crs.iterrows():

    point = row.geometry.representative_point()

    ax.text(
        point.x,
        point.y,
        str(row["polygon_id"]),
        fontsize=7,
        ha="center",
        va="center",
        color="white",
        bbox=dict(
            facecolor="black",
            alpha=0.65,
            edgecolor="none",
            pad=1
        )
    )

plt.colorbar(
    image,
    ax=ax,
    label="NDVI"
)

ax.set_title(
    f"NDVI with Soil–Vegetation Polygon IDs — "
    f"{first_ndvi_date:%Y-%m-%d}"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_31.png", dpi=200, bbox_inches="tight")
plt.show()

## 17. Inspect the selected agricultural polygons

The selected polygons are inspected to identify their soil type, vegetation class, area, and other attributes. This establishes exactly which mapped regions will be analyzed in the NDVI time series.

In [ ]:
crop_polygon_ids = [12, 43, 65, 71, 95]

crop_fields = soil_vegetation_ndvi_crs[
    soil_vegetation_ndvi_crs["polygon_id"].isin(crop_polygon_ids)
].copy()

crop_fields[
    [
        "polygon_id",
        "soil_name",
        "area_km2",
        "LCNAME",
        "NEWVEG",
        "TNC_CLASS",
        "SPECIES_CO",
        "DISTURBED_"
    ]
].sort_values("polygon_id")

## 18. Show only the combined agricultural region

The five selected agricultural polygons are highlighted on the NDVI raster. This provides a visual check that the selected IDs correspond to the intended agricultural areas.

In [ ]:
with rasterio.open(first_ndvi_path) as src:

    ndvi = src.read(1, masked=True)
    bounds = src.bounds

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

fig, ax = plt.subplots(figsize=(12, 8))

# NDVI background
image = ax.imshow(
    ndvi,
    extent=extent,
    origin="upper",
    vmin=-1,
    vmax=1,
    cmap="viridis"
)

# All soil–vegetation polygons
soil_vegetation_ndvi_crs.boundary.plot(
    ax=ax,
    linewidth=0.4,
    color="black",
    alpha=0.35
)

# Selected crop fields
crop_fields.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2
)

# Labels
for _, row in crop_fields.iterrows():

    point = row.geometry.representative_point()

    ax.text(
        point.x,
        point.y,
        str(row["polygon_id"]),
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="center",
        color="white",
        bbox=dict(
            facecolor="red",
            alpha=0.8,
            edgecolor="none",
            pad=2
        )
    )

plt.colorbar(
    image,
    ax=ax,
    label="NDVI"
)

ax.set_title(
    f"Selected Crop Fields — NDVI {first_ndvi_date:%Y-%m-%d}"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_35.png", dpi=200, bbox_inches="tight")
plt.show()

## 19. Extract NDVI statistics for the selected agricultural polygons

NDVI statistics are calculated separately for each selected agricultural-region polygon and each acquisition date. The analysis uses the actual polygon geometry, rather than the entire AOI, so the resulting time series represents the selected agricultural fields.

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

crop_ndvi_records = []

for _, polygon in crop_fields.iterrows():

    polygon_id = polygon["polygon_id"]

    geometry = [polygon.geometry]

    for _, raster_row in ndvi_inventory.iterrows():

        with rasterio.open(raster_row["path"]) as src:

            ndvi, _ = mask(
                src,
                geometry,
                crop=True,
                filled=False
            )

            values = ndvi[0].compressed()

            crop_ndvi_records.append({
                "polygon_id": polygon_id,
                "acquisition_date": raster_row["acquisition_date"],
                "valid_pixels": len(values),
                "mean_ndvi": float(np.mean(values)),
                "median_ndvi": float(np.median(values)),
                "min_ndvi": float(np.min(values)),
                "max_ndvi": float(np.max(values)),
                "std_ndvi": float(np.std(values))
            })

crop_ndvi_stats = (
    pd.DataFrame(crop_ndvi_records)
    .sort_values(["polygon_id", "acquisition_date"])
    .reset_index(drop=True)
)

display(crop_ndvi_stats)

crop_ndvi_stats.to_csv(TABLES_DIR / "stage5_agricultural_polygon_ndvi_statistics.csv", index=False)

## 20. Combine the selected agricultural polygons into one analysis region


The selected agricultural polygons are combined into a single geometry. The polygons themselves do not overlap, so this union represents the total mapped agricultural area without double-counting pixels where polygon boundaries meet.

In [ ]:
from shapely.ops import unary_union
import geopandas as gpd

crop_polygon_ids = [12, 43, 65, 71, 95]

crop_fields = soil_vegetation_ndvi_crs[
    soil_vegetation_ndvi_crs["polygon_id"].isin(crop_polygon_ids)
].copy()

crop_geometry = unary_union(crop_fields.geometry)

crop_region = gpd.GeoDataFrame(
    {
        "region": ["Selected crop fields"],
        "area_km2": [crop_geometry.area / 1_000_000]
    },
    geometry=[crop_geometry],
    crs=soil_vegetation_ndvi_crs.crs
)

crop_region

## 21. Verify the combined agricultural region

The combined geometry is displayed to verify that all five selected agricultural areas are included and that the union has not introduced unexpected geometry.

In [ ]:
with rasterio.open(first_ndvi_path) as src:

    ndvi = src.read(1, masked=True)
    bounds = src.bounds

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

fig, ax = plt.subplots(figsize=(12, 8))

image = ax.imshow(
    ndvi,
    extent=extent,
    origin="upper",
    vmin=-1,
    vmax=1,
    cmap="viridis"
)

# All soil–vegetation boundaries
soil_vegetation_ndvi_crs.boundary.plot(
    ax=ax,
    linewidth=0.4,
    color="black",
    alpha=0.3
)

# Combined crop-field region
crop_region.boundary.plot(
    ax=ax,
    linewidth=2,
    color="red"
)

plt.colorbar(
    image,
    ax=ax,
    label="NDVI"
)

ax.set_title(
    f"Selected Crop-Field Region — {first_ndvi_date:%Y-%m-%d}"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_41.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------
# Inspect NDVI and SCL raster filenames
# ---------------------------------------------------------

from pathlib import Path

satellite_dir = PROJECT_DIR / "data" / "processed" / "satellite"

ndvi_aoi_dir = satellite_dir / "ndvi_aoi"
ndvi_clipped_dir = ndvi_aoi_dir / "clipped"

print("=== NDVI rasters: ndvi_aoi ===")

for path in sorted(ndvi_aoi_dir.glob("*.tif")):
    print(path.name)


print("\n=== NDVI rasters: ndvi_aoi/clipped ===")

for path in sorted(ndvi_clipped_dir.glob("*.tif")):
    print(path.name)


print("\n=== SCL rasters ===")

scl_files = [
    path
    for path in satellite_dir.rglob("*.tif")
    if "SCL" in path.name.upper()
]

for path in sorted(scl_files):
    print(path.relative_to(satellite_dir))

In [ ]:
# ---------------------------------------------------------
# Inspect Sentinel-2 SCL files
# ---------------------------------------------------------

from pathlib import Path

satellite_dir = PROJECT_DIR / "data" / "raw" / "satellite" / "sentinel2"

scl_files = [
    path
    for path in satellite_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".jp2", ".tif"}
    and "SCL" in path.name.upper()
]

print(f"Number of SCL files found: {len(scl_files)}\n")

for path in sorted(scl_files):
    print(path.relative_to(satellite_dir))

In [ ]:
# ---------------------------------------------------------
# Inspect all Sentinel-2 raster files
# ---------------------------------------------------------

raster_files = [
    path
    for path in satellite_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".jp2", ".tif"}
]

print(f"Total raster files: {len(raster_files)}\n")

for path in sorted(raster_files):
    print(path.relative_to(satellite_dir))
    

## 22. Create the output directory and build the NDVI inventory

This cell collects all NDVI GeoTIFFs from both existing folders, extracts their acquisition dates, and prepares the output directory. We keep the source folder in the inventory because the same date could potentially exist in both locations.

In [ ]:
# ---------------------------------------------------------
# Build NDVI inventory and prepare crop-region output
# ---------------------------------------------------------

from pathlib import Path
import re
import pandas as pd

satellite_out = PROJECT_DIR / "data" / "processed" / "satellite"

ndvi_aoi_dir = satellite_out / "ndvi_aoi"
ndvi_clipped_dir = ndvi_aoi_dir / "clipped"

# New directory for the selected crop-field region
ndvi_crop_dir = satellite_out / "ndvi_crop_region"
ndvi_crop_dir.mkdir(exist_ok=True)

ndvi_folders = [
    ndvi_aoi_dir,
    ndvi_clipped_dir,
]

ndvi_records = []

for folder in ndvi_folders:

    for path in folder.glob("*.tif"):

        match = re.search(r"\d{4}-\d{2}-\d{2}", path.name)

        if match is None:
            raise ValueError(
                f"No acquisition date found in: {path.name}"
            )

        ndvi_records.append({
            "acquisition_date": pd.to_datetime(match.group()),
            "file": path.name,
            "path": path,
            "source_folder": folder.name,
        })

ndvi_inventory = (
    pd.DataFrame(ndvi_records)
    .sort_values(["acquisition_date", "file"])
    .reset_index(drop=True)
)

print(f"NDVI rasters found: {len(ndvi_inventory)}")
print(f"Output directory: {ndvi_crop_dir}")

display(ndvi_inventory.head())
ndvi_inventory.to_csv(TABLES_DIR / "stage5_ndvi_inventory_for_scl_screening.csv", index=False)

## 23. Match Each NDVI Acquisition to Its SCL Raster

The NDVI files and SCL files use the same acquisition date and Sentinel-2 tile identifier. We will use those two pieces of information to establish the correspondence between each NDVI raster and its SCL raster. This avoids relying on the position of files in a directory and gives us an explicit, reproducible match for cloud screening.

In [ ]:
# ---------------------------------------------------------
# Build SCL inventory and match it to the NDVI inventory
# ---------------------------------------------------------

scl_files = [
    path
    for path in satellite_dir.rglob("*.jp2")
    if "SCL" in path.name.upper()
]

scl_records = []

for path in scl_files:

    match = re.search(
        r"(\d{4}-\d{2}-\d{2}).*?_(17[A-Z]{3})_SCL",
        path.name
    )

    if match is None:
        print(f"Could not parse SCL filename: {path.name}")
        continue

    scl_records.append({
        "acquisition_date": pd.to_datetime(match.group(1)),
        "tile": match.group(2),
        "scl_path": path,
        "scl_file": path.name,
    })

scl_inventory = pd.DataFrame(scl_records)

print(f"SCL rasters found: {len(scl_inventory)}")
scl_inventory.head()

## 24. Verify NDVI–SCL Acquisition Matches

Before processing the rasters, we verify which Sentinel-2 tiles are available for each acquisition date. This is important because the NDVI products can represent a mosaic of `17QLF` and `17QMF`, while the SCL information exists separately for each tile.

In [ ]:
# ---------------------------------------------------------
# Inspect SCL coverage by acquisition date
# ---------------------------------------------------------

scl_by_date = (
    scl_inventory
    .groupby("acquisition_date")["tile"]
    .apply(list)
    .reset_index(name="tiles")
)

print(scl_by_date.to_string(index=False))

## 25. Crop NDVI Rasters to the Selected Crop Fields

Each NDVI raster will be masked with the combined `crop_geometry`. The result is a new GeoTIFF containing only the pixels inside the combined agricultural region. The original NDVI files remain untouched, allowing us to reproduce or modify the workflow later.

In [ ]:
# ---------------------------------------------------------
# Crop NDVI rasters to the selected crop-field region
# ---------------------------------------------------------

import rasterio
from rasterio.mask import mask

for _, row in ndvi_inventory.iterrows():

    input_path = row["path"]

    output_path = (
        ndvi_crop_dir /
        f"{input_path.stem}_crop_region.tif"
    )

    with rasterio.open(input_path) as src:

        ndvi_crop, crop_transform = mask(
            src,
            [crop_geometry],
            crop=True,
            filled=True,
            nodata=-9999.0
        )

        profile = src.profile.copy()

        profile.update({
            "height": ndvi_crop.shape[1],
            "width": ndvi_crop.shape[2],
            "transform": crop_transform,
            "driver": "GTiff",
            "dtype": "float32",
            "nodata": -9999.0,
            "compress": "deflate",
        })

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(ndvi_crop.astype("float32"))

print(
    f"Created {len(ndvi_inventory)} crop-region NDVI rasters."
)
print(f"Saved to: {ndvi_crop_dir}")

## 26. Calculate Cloud Coverage Over the Selected Crop Fields

For each NDVI acquisition, this step finds all corresponding Sentinel-2 SCL rasters, crops them to the same `crop_geometry`, and combines the resulting SCL data when more than one tile covers the crop region. Cloud coverage is calculated only inside the combined agricultural region.

SCL classes 8, 9, and 10 are counted as cloud/cirrus. Class 3 (cloud shadow) is reported separately because it can also contaminate NDVI even though it is not technically cloud.

In [ ]:
# ---------------------------------------------------------
# Calculate cloud and cloud-shadow percentage
# over the selected crop-field region
# ---------------------------------------------------------

import numpy as np
import rasterio
from rasterio.mask import mask

# Sentinel-2 SCL classes
CLOUD_CLASSES = {8, 9, 10}
CLOUD_SHADOW_CLASS = 3

cloud_records = []

for date in sorted(ndvi_inventory["acquisition_date"].unique()):

    # Find all SCL rasters for this acquisition
    scl_matches = scl_inventory[
        scl_inventory["acquisition_date"] == date
    ]

    if scl_matches.empty:
        print(f"No SCL raster found for {date:%Y-%m-%d}")
        continue

    total_valid = 0
    total_cloud = 0
    total_shadow = 0

    for _, scl_row in scl_matches.iterrows():

        with rasterio.open(scl_row["scl_path"]) as src:

            scl_crop, _ = mask(
                src,
                [crop_geometry],
                crop=True,
                filled=False
            )

        scl_values = scl_crop[0].compressed()

        if len(scl_values) == 0:
            continue

        total_valid += len(scl_values)

        total_cloud += np.isin(
            scl_values,
            list(CLOUD_CLASSES)
        ).sum()

        total_shadow += (
            scl_values == CLOUD_SHADOW_CLASS
        ).sum()

    if total_valid == 0:
        continue

    cloud_pct = (
        total_cloud / total_valid
    ) * 100

    cloud_shadow_pct = (
        total_shadow / total_valid
    ) * 100

    cloud_records.append({
        "acquisition_date": date,
        "valid_scl_pixels": total_valid,
        "cloud_pixels": total_cloud,
        "cloud_pct": cloud_pct,
        "cloud_shadow_pixels": total_shadow,
        "cloud_shadow_pct": cloud_shadow_pct,
    })

cloud_screening = (
    pd.DataFrame(cloud_records)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

display(cloud_screening)

cloud_screening.to_csv(TABLES_DIR / "stage5_cloud_screening.csv", index=False)

## 27. Select Acquisitions With Less Than 10% Cloud

The cloud-screening criterion is applied to the **combined agricultural region only**. Acquisitions with `cloud_pct < 10` are retained for the subsequent NDVI analysis. Cloud-shadow percentage remains visible as an additional quality-control variable rather than being silently included in the cloud percentage.

In [ ]:
# ---------------------------------------------------------
# Select acquisitions with cloud percentage < 10%
# ---------------------------------------------------------

selected_cloud_screening = (
    cloud_screening[
        cloud_screening["cloud_pct"] < 10
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Selected acquisitions: "
    f"{len(selected_cloud_screening)}"
)

display(selected_cloud_screening)
selected_cloud_screening.to_csv(TABLES_DIR / "stage5_selected_low_cloud_acquisitions.csv", index=False)

## 28. Visual Inspection of Low-Cloud NDVI Acquisitions

This cell plots all NDVI rasters that passed the `<10%` cloud filter in a grid. Each panel shows the combined agricultural region using the same NDVI scale, allowing visual comparison between acquisition dates and identification of remaining cloud, shadow, or other spatial artifacts.

In [ ]:
# ---------------------------------------------------------
# Build inventory of crop-region NDVI rasters
# ---------------------------------------------------------

crop_ndvi_records = []

for path in sorted(ndvi_crop_dir.glob("*.tif")):

    match = re.search(
        r"\d{4}-\d{2}-\d{2}",
        path.name
    )

    if match is None:
        raise ValueError(
            f"No acquisition date found in: {path.name}"
        )

    crop_ndvi_records.append({
        "acquisition_date": pd.to_datetime(match.group()),
        "path": path,
        "file": path.name
    })

crop_ndvi_inventory = (
    pd.DataFrame(crop_ndvi_records)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

print(
    f"Crop-region NDVI rasters: "
    f"{len(crop_ndvi_inventory)}"
)

crop_ndvi_inventory.head()

In [ ]:
# ---------------------------------------------------------
# Plot NDVI rasters selected by the <10% cloud filter
# ---------------------------------------------------------

import math
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import plotting_extent

# ---------------------------------------------------------
# Select acquisitions with cloud percentage < 10%
# ---------------------------------------------------------

selected_cloud_screening = (
    cloud_screening[
        cloud_screening["cloud_pct"] < 10
    ]
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

print(
    f"Number of acquisitions selected: "
    f"{len(selected_cloud_screening)}"
)

# ---------------------------------------------------------
# Number of rows and columns
# ---------------------------------------------------------

n_maps = len(selected_cloud_screening)

ncols = 3
nrows = math.ceil(n_maps / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(15, 4.5 * nrows),
    constrained_layout=True
)

# Make axes iterable even if there is only one subplot
axes = np.atleast_1d(axes).ravel()

# ---------------------------------------------------------
# Plot each selected raster
# ---------------------------------------------------------

for ax, (_, row) in zip(
    axes,
    selected_cloud_screening.iterrows()
):

    date = row["acquisition_date"]

    # Find the cropped NDVI raster for this date
    matches = crop_ndvi_inventory[
        crop_ndvi_inventory["acquisition_date"] == date
    ]

    if matches.empty:
        ax.set_title(
            f"{date:%Y-%m-%d}\nNDVI raster not found"
        )
        ax.axis("off")
        continue

    # Use the first matching raster
    # if duplicate source folders contain the same date.
    ndvi_path = matches.iloc[0]["path"]

    with rasterio.open(ndvi_path) as src:

        ndvi = src.read(1)
        extent = plotting_extent(ndvi, src.transform)

        # Convert NoData to NaN
        if src.nodata is not None:
            ndvi = np.where(
                ndvi == src.nodata,
                np.nan,
                ndvi
            )

    image = ax.imshow(
        ndvi,
        extent=extent,
        origin="upper",
        vmin=-1,
        vmax=1,
        cmap="viridis"
    )

    # Crop-field boundary
    crop_region.boundary.plot(
        ax=ax,
        color="white",
        linewidth=1.2
    )

    ax.set_title(
        f"{date:%Y-%m-%d}\n"
        f"Cloud: {row['cloud_pct']:.1f}%"
    )

    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_aspect("equal")

# ---------------------------------------------------------
# Hide unused panels
# ---------------------------------------------------------

for ax in axes[n_maps:]:
    ax.axis("off")

# ---------------------------------------------------------
# Shared colorbar
# ---------------------------------------------------------

fig.colorbar(
    image,
    ax=axes[:n_maps],
    label="NDVI",
    shrink=0.85
)

fig.suptitle(
    "Visual Inspection of NDVI Acquisitions — Cloud < 10%",
    fontsize=16
)

plt.savefig(FIGURES_DIR / "stage5_figure_59.png", dpi=200, bbox_inches="tight")
plt.show()

## 29. Build the Final NDVI Analysis Inventory

The `<10%` cloud threshold is now treated as the final quality-control criterion. This step combines the cloud-screening results with the agricultural-region NDVI rasters so that every raster used in the analysis is both spatially restricted to the combined agricultural region and associated with an acquisition meeting the cloud criterion.

In [ ]:
# ---------------------------------------------------------
# Build final NDVI analysis inventory
# Cloud threshold: <10%
# ---------------------------------------------------------

cloud_threshold = 10

selected_acquisitions = (
    cloud_screening[
        cloud_screening["cloud_pct"] < cloud_threshold
    ]
    .copy()
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

# Match selected dates to the crop-region NDVI rasters
final_ndvi_inventory = crop_ndvi_inventory.merge(
    selected_acquisitions[
        [
            "acquisition_date",
            "cloud_pct",
            "cloud_shadow_pct",
        ]
    ],
    on="acquisition_date",
    how="inner",
)

final_ndvi_inventory = (
    final_ndvi_inventory
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

print(
    f"Final NDVI acquisitions: "
    f"{len(final_ndvi_inventory)}"
)

final_ndvi_inventory[
    [
        "acquisition_date",
        "cloud_pct",
        "cloud_shadow_pct",
        "file",
    ]
]


display(final_ndvi_inventory)

final_ndvi_inventory.to_csv(TABLES_DIR / "stage5_final_ndvi_inventory.csv", index=False)

## 30. Calculate Descriptive Statistics for Each Acquisition

For every selected NDVI acquisition, the pixels inside the combined agricultural region are extracted and summarized. Mean and median describe the central NDVI level, standard deviation describes spatial variability, and minimum/maximum show the observed range. The number of valid pixels provides a useful consistency check across dates.

In [ ]:
# ---------------------------------------------------------
# Calculate NDVI descriptive statistics
# ---------------------------------------------------------

import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

ndvi_stats_records = []

for _, row in final_ndvi_inventory.iterrows():

    with rasterio.open(row["path"]) as src:

        ndvi, _ = mask(
            src,
            [crop_geometry],
            crop=True,
            filled=False
        )

    values = ndvi[0].compressed()

    ndvi_stats_records.append({
        "acquisition_date": row["acquisition_date"],
        "cloud_pct": row["cloud_pct"],
        "cloud_shadow_pct": row["cloud_shadow_pct"],
        "pixel_count": len(values),
        "mean_ndvi": np.mean(values),
        "median_ndvi": np.median(values),
        "std_ndvi": np.std(values),
        "min_ndvi": np.min(values),
        "max_ndvi": np.max(values),
    })

crop_ndvi_stats = (
    pd.DataFrame(ndvi_stats_records)
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

display(crop_ndvi_stats)
crop_ndvi_stats.to_csv(TABLES_DIR / "stage5_agricultural_region_ndvi_statistics.csv", index=False)

## 31. Check Spatial Consistency Across Acquisitions

Because every acquisition is analyzed over the same agricultural-region geometry, the number of valid pixels should be broadly consistent. Large differences would indicate a spatial or NoData issue that should be investigated before interpreting the time series.

In [ ]:
# ---------------------------------------------------------
# Check valid pixel counts
# ---------------------------------------------------------

print(
    crop_ndvi_stats[
        ["acquisition_date", "pixel_count"]
    ].to_string(index=False)
)

print(
    "\nUnique pixel counts:",
    crop_ndvi_stats["pixel_count"].unique()
)

## 32. Temporal NDVI Line Plot

The line plot shows how the central NDVI value of the combined agricultural region changes through time. Mean NDVI represents the average vegetation signal, while median NDVI provides a second measure that is less affected by unusually low or high pixels.

In [ ]:
# ---------------------------------------------------------
# Plot mean and median NDVI through time
# ---------------------------------------------------------

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    crop_ndvi_stats["acquisition_date"],
    crop_ndvi_stats["mean_ndvi"],
    marker="o",
    label="Mean NDVI"
)

ax.plot(
    crop_ndvi_stats["acquisition_date"],
    crop_ndvi_stats["median_ndvi"],
    marker="o",
    linestyle="--",
    label="Median NDVI"
)

ax.set_title(
    "NDVI Through Time — Selected Crop Fields"
)

ax.set_xlabel("Acquisition date")
ax.set_ylabel("NDVI")

ax.set_ylim(-1, 1)

ax.grid(
    True,
    alpha=0.3
)

ax.legend()

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(FIGURES_DIR / "stage5_figure_67.png", dpi=200, bbox_inches="tight")
plt.show()

## 33. NDVI Distribution by Acquisition Date

The boxplot shows the spatial distribution of NDVI pixels within the combined agricultural region for each acquisition. The median is the central line, the box represents the middle 50% of observations, and the whiskers show the broader distribution. Outlier markers are hidden to keep the temporal comparison readable.

In [ ]:
# ---------------------------------------------------------
# Prepare NDVI distributions for boxplot
# ---------------------------------------------------------

boxplot_data = []
boxplot_dates = []

for _, row in final_ndvi_inventory.iterrows():

    with rasterio.open(row["path"]) as src:

        ndvi, _ = mask(
            src,
            [crop_geometry],
            crop=True,
            filled=False
        )

    values = ndvi[0].compressed()

    boxplot_data.append(values)
    boxplot_dates.append(
        row["acquisition_date"]
    )

In [ ]:
# ---------------------------------------------------------
# Plot NDVI distributions
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 7))

ax.boxplot(
    boxplot_data,
    tick_labels=[
        date.strftime("%Y-%m-%d")
        for date in boxplot_dates
    ],
    showfliers=False
)

ax.set_title(
    "NDVI Distribution Across Selected Crop Fields"
)

ax.set_xlabel("Acquisition date")
ax.set_ylabel("NDVI")

ax.set_ylim(-1, 1)

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(FIGURES_DIR / "stage5_figure_70.png", dpi=200, bbox_inches="tight")
plt.show()

## 34. Inspect Available Dates Before Selecting Density Curves

The density plot is most useful when a small number of representative acquisitions are compared. This cell displays the dates that survived the `<10%` cloud filter so we can choose representative dates from the actual final dataset rather than assuming that specific dates are available.

In [ ]:
# ---------------------------------------------------------
# Display final acquisition dates
# ---------------------------------------------------------

for date in final_ndvi_inventory["acquisition_date"]:
    print(date.strftime("%Y-%m-%d"))

## 35. Compare NDVI Density Distributions

The density plot shows the shape of the NDVI distribution within the combined agricultural region. Unlike the line plot, which summarizes each date with a central value, this visualization shows whether the field is relatively homogeneous or contains multiple concentrations of NDVI values.

In [ ]:
# ---------------------------------------------------------
# NDVI density distributions for representative dates
# ---------------------------------------------------------

from scipy.stats import gaussian_kde

# Replace these with four dates from the final inventory
density_dates = [
    pd.Timestamp("2025-12-08"),
    pd.Timestamp("2026-02-16"),
    pd.Timestamp("2026-04-29"),
    pd.Timestamp("2026-06-16"),
]

fig, ax = plt.subplots(figsize=(11, 7))

for date in density_dates:

    matches = final_ndvi_inventory[
        final_ndvi_inventory["acquisition_date"] == date
    ]

    if matches.empty:
        print(
            f"Date not available: {date:%Y-%m-%d}"
        )
        continue

    raster_path = matches.iloc[0]["path"]

    with rasterio.open(raster_path) as src:

        ndvi, _ = mask(
            src,
            [crop_geometry],
            crop=True,
            filled=False
        )

    values = ndvi[0].compressed()

    kde = gaussian_kde(values)

    x = np.linspace(-1, 1, 500)

    density = kde(x)

    ax.plot(
        x,
        density,
        label=date.strftime("%Y-%m-%d")
    )

ax.set_title(
    "NDVI Density Distribution Across Selected Dates"
)

ax.set_xlabel("NDVI")
ax.set_ylabel("Density")

ax.set_xlim(-1, 1)

ax.grid(
    True,
    alpha=0.3
)

ax.legend(
    title="Acquisition date"
)

plt.tight_layout()

plt.savefig(FIGURES_DIR / "stage5_figure_74.png", dpi=200, bbox_inches="tight")
plt.show()

## 36. Spatial Distribution of NDVI

The maps show the spatial pattern of NDVI within the combined agricultural region for representative acquisition dates. All panels use the same NDVI scale (`-1` to `1`), making spatial differences between dates directly comparable. The agricultural-region boundary is displayed to make the analysis region explicit.

In [ ]:
# ---------------------------------------------------------
# 2×2 NDVI maps for representative dates
# ---------------------------------------------------------

from rasterio.plot import plotting_extent

representative_dates = [
    pd.Timestamp("2025-12-08"),
    pd.Timestamp("2026-02-16"),
    pd.Timestamp("2026-04-29"),
    pd.Timestamp("2026-06-16"),
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 11),
    constrained_layout=True
)

axes = axes.ravel()

for ax, map_date in zip(
    axes,
    representative_dates
):

    matches = final_ndvi_inventory[
        final_ndvi_inventory["acquisition_date"] == map_date
    ]

    if matches.empty:
        ax.set_title(
            f"NDVI — {map_date:%Y-%m-%d}\n"
            "Date not available"
        )
        ax.axis("off")
        continue

    raster_path = matches.iloc[0]["path"]

    with rasterio.open(raster_path) as src:

        ndvi = src.read(1)

        if src.nodata is not None:
            ndvi = np.where(
                ndvi == src.nodata,
                np.nan,
                ndvi
            )

        extent = plotting_extent(
            ndvi,
            src.transform
        )

    image = ax.imshow(
        ndvi,
        extent=extent,
        origin="upper",
        vmin=-1,
        vmax=1,
        cmap="viridis"
    )

    crop_region.boundary.plot(
        ax=ax,
        color="white",
        linewidth=1.5
    )

    ax.set_title(
        f"NDVI — {map_date:%Y-%m-%d}"
    )

    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_aspect("equal")

fig.colorbar(
    image,
    ax=axes,
    label="NDVI",
    shrink=0.85
)

fig.suptitle(
    "NDVI Spatial Distribution Across Selected Crop Fields",
    fontsize=16
)

plt.savefig(FIGURES_DIR / "stage5_figure_76.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Map NDVI for one selected acquisition date

import numpy as np
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent

#map_date = pd.Timestamp("2026-04-29")
#map_date = pd.Timestamp("2025-12-08")
#map_date = pd.Timestamp("2026-02-16")
#map_date = pd.Timestamp("2026-05-07")
map_date = pd.Timestamp("2026-06-16")

# Find the raster for the selected date
matches = final_ndvi_inventory[
    final_ndvi_inventory["acquisition_date"] == map_date
]

if matches.empty:
    raise ValueError(f"No NDVI raster found for {map_date:%Y-%m-%d}")

raster_path = matches.iloc[0]["path"]

# Read the NDVI raster
with rasterio.open(raster_path) as src:
    ndvi = src.read(1)
    transform = src.transform
    nodata = src.nodata

# Convert NoData pixels to NaN
if nodata is not None:
    ndvi = np.where(ndvi == nodata, np.nan, ndvi)

# Calculate raster extent for plotting
extent = plotting_extent(ndvi, transform)

# Create the map
fig, ax = plt.subplots(figsize=(10, 8))

image = ax.imshow(
    ndvi,
    extent=extent,
    origin="upper",
    vmin=-1,
    vmax=1,
    cmap="viridis"
)

# Add the boundary of the selected crop-field region
crop_region.boundary.plot(
    ax=ax,
    color="white",
    linewidth=1.5
)

# Labels and title
ax.set_title(
    f"NDVI — Selected Crop Fields\n{map_date:%Y-%m-%d}",
    fontsize=14
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.set_aspect("equal")

# NDVI legend
cbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.85
)

cbar.set_label("NDVI")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_77.png", dpi=200, bbox_inches="tight")
plt.show()

## 37. Agricultural Region NDVI Time Series

The sample agricultural polygons are used to extract NDVI statistics
from the selected Sentinel-2 observations.

The resulting table contains one observation for each combination of
agricultural field and acquisition date.

In [ ]:
from pathlib import Path
import pandas as pd

# Path to the agricultural-region NDVI time-series table
project_root = PROJECT_DIR / "data" / "processed"

crop_ndvi_path = (
    project_root
    / "tables"
    / "crop_ndvi_timeseries.csv"
)

crop_ndvi = pd.read_csv(crop_ndvi_path)

crop_ndvi.head()

### 37.1 Prepare the observation dates

The acquisition date is converted to a datetime field so the observations
can be sorted and plotted chronologically.

In [ ]:
crop_ndvi["date"] = pd.to_datetime(crop_ndvi["date"])

crop_ndvi =crop_ndvi.sort_values(
    ["field_id", "date"]
).reset_index(drop=True)

crop_ndvi.info()

In [ ]:
crop_ndvi[
    [
        "field_id",
        "crop_type",
        "date",
        "platform",
        "valid_pixels",
        "mean_ndvi",
        "median_ndvi",
        "std_ndvi",
    ]
]

## 38. Agricultural Region NDVI Temporal Profile

Mean NDVI is plotted through time for the whole sampled agricultural fields.
The purpose is to examine the temporal vegetation signal during the
study period.

In [ ]:
# Line plot — combined agricultural region

combined_ndvi = (
    crop_ndvi
    .groupby("date", as_index=False)["mean_ndvi"]
    .mean()
    .sort_values("date")
)

plt.figure(figsize=(11, 5))

plt.plot(
    combined_ndvi["date"],
    combined_ndvi["mean_ndvi"],
    marker="o"
)

plt.xlabel("Date")
plt.ylabel("Mean NDVI")
plt.title("Agricultural Region NDVI Through Time")
plt.ylim(-1, 1)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_84.png", dpi=200, bbox_inches="tight")
plt.show()

## 39. Agricultural Region NDVI Distribution

The distribution of mean NDVI across all observations from the two
sampled agricultural areas is summarized using a boxplot.

In [ ]:
# Boxplot — NDVI distribution for each observation date

boxplot_data = [
    crop_ndvi.loc[
        crop_ndvi["date"] == date,
        "mean_ndvi"
    ].dropna()
    for date in sorted(crop_ndvi["date"].unique())
]

date_labels = [
    pd.Timestamp(date).strftime("%b %d")
    for date in sorted(crop_ndvi["date"].unique())
]

plt.figure(figsize=(13, 5))

plt.boxplot(
    boxplot_data,
    tick_labels=date_labels
)

plt.xlabel("Observation Date")
plt.ylabel("Mean NDVI")
plt.title("NDVI Distribution by Observation Date — Agricultural Region")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_86.png", dpi=200, bbox_inches="tight")
plt.show()

## 40. Agricultural Region NDVI Density

The distribution of mean NDVI values is visualized as a kernel density
estimate to examine the concentration and shape of the observations.

In [ ]:
# Select four representative dates for the density analysis

selected_dates = [
    "2025-11-15",
    "2026-01-24",
    "2026-03-15",
    "2026-04-29",
]

ndvi_crop_dir = (
    Path(project_root)
    / "satellite"
    / "ndvi_crop_region"
)

raster_files = []

for date in selected_dates:
    matches = list(ndvi_crop_dir.glob(f"{date}_*_NDVI_AOI_crop_region.tif"))
    print(date, matches)
    raster_files.extend(matches)

In [ ]:
# Extract pixel-level NDVI values from both potato fields

import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Path to the potato field polygons
crop_samples_path = (
    Path(project_root)
    / "vector"
    / "crop_field_samples.gpkg"
)

crop_fields = gpd.read_file(
    crop_samples_path,
    layer="potato_field_samples"
)

print(crop_fields)
print("CRS:", crop_fields.crs)

In [ ]:
# Extract NDVI pixels for both potato fields at each selected date

density_data = {}

for date in selected_dates:

    raster_path = list(
        ndvi_crop_dir.glob(
            f"{date}_*_NDVI_AOI_crop_region.tif"
        )
    )[0]

    with rasterio.open(raster_path) as src:

        # Reproject polygons to raster CRS if necessary
        fields = crop_fields.to_crs(src.crs)

        # Read raster data
        ndvi = src.read(1)

        # Raster transform
        transform = src.transform

        # Create masks for both fields
        from rasterio.features import geometry_mask

        combined_mask = np.zeros(
            (src.height, src.width),
            dtype=bool
        )

        for geometry in fields.geometry:

            field_mask = geometry_mask(
                [geometry],
                transform=transform,
                invert=True,
                out_shape=(src.height, src.width)
            )

            combined_mask |= field_mask

        # Extract pixels inside either potato field
        values = ndvi[combined_mask]

        # Remove invalid / NoData values
        values = values[np.isfinite(values)]

        # NDVI should normally be within this physical range
        values = values[(values >= -1) & (values <= 1)]

        density_data[date] = values

        print(
            f"{date}: {len(values):,} valid pixels"
        )

In [ ]:
# NDVI density distribution across four representative dates

plt.figure(figsize=(10, 6))

for date, values in density_data.items():

    pd.Series(values).plot(
        kind="density",
        linewidth=2,
        label=pd.Timestamp(date).strftime("%b %d, %Y")
    )

plt.xlabel("NDVI")
plt.ylabel("Density")
plt.title("NDVI Density Distribution Across Selected Dates")
plt.xlim(-1, 1)
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_figure_91.png", dpi=200, bbox_inches="tight")
plt.show()

## Workflow Summary

This notebook analyzes Sentinel-2 NDVI observations for the GeoSoil-Cuba Jaruco/Mayabeque area of interest (AOI). It validates the available rasters, examines spatial and temporal NDVI patterns, screens acquisitions using SCL cloud information over a combined agricultural region, and prepares tables and figures for later reporting.

### Project setup and source data

- Set `PROJECT_DIR = Path("..").resolve()` and defined paths for the processed NDVI rasters, soil–vegetation GeoPackage, and report outputs.
- Used `soil_vegetation_analysis.gpkg` as the vector source for the AOI and agricultural polygon context.
- Kept the source NDVI GeoTIFFs separate from derived agricultural-region rasters and analysis outputs.

### NDVI validation and exploratory analysis

- Checked raster metadata, spatial grids, acquisition dates, temporal coverage, and basic NDVI statistics.
- Inspected extreme NDVI values and compared mean, median, distribution, and spatial patterns through time.
- Overlaid the soil–vegetation polygons on NDVI maps and identified the selected agricultural polygons for focused analysis.

### Agricultural-region cloud screening and time series

- Combined the selected agricultural polygons into one analysis region.
- Matched NDVI observations to corresponding Sentinel-2 SCL rasters and estimated cloud and cloud-shadow coverage over the region.
- Retained observations meeting the `<10%` cloud criterion for subsequent summaries and visualizations.
- Calculated descriptive statistics and generated temporal, distribution, density, and spatial plots for the combined agricultural region.

### Outputs for reports and presentations

- Created `outputs/tables` and `outputs/figures` for reusable CSV tables and high-resolution plot files.
- Saved key analysis tables and figures so they can be incorporated into future PDF reports and PowerPoint presentations.

### Stage 5 conclusions

This stage turns the available AOI-level NDVI rasters into a validated and visually interpretable time series, with additional quality screening over a combined agricultural analysis region. The retained observations and saved tables and figures provide a reproducible basis for later crop-specific interpretation and portfolio reporting. Crop-specific labels can be selected later when the agricultural target has been finalized.